# 2D → 3D Pipeline — Colab **Run all** (P6 - NVIDIA + Depth-Anything-V2)

**Runtime ▸ Change runtime type ▸ T4 GPU** → **Runtime ▸ Run all**. Xong, không cần bấm gì thêm.

Hệ thống vận hành chuẩn xác 2 kịch bản:
- **Kịch bản 1: Đơn ảnh (1 ảnh)** → Depth-Anything-V2 Pinhole Surface Mesh / TripoSR.
- **Kịch bản 2: Đa ảnh (N ảnh, N ≥ 2)** → Depth-Anything-V2 + NVIDIA Volumetric TSDF & Marching Cubes (đặc ruột, kín nước 100%).

| Cell | Việc |
| :--: | :--- |
| 1 | Clone repo (nhánh P6-FullStack-Cloud) + cài dependencies + Depth-Anything-V2 + TripoSR |
| 2 | Chọn bộ dữ liệu ảnh: 'objaverse' (6 ảnh 360° quả táo) hoặc 'gso' (5 ảnh giày thể thao) |
| 3 | Khởi động FastAPI server + Cloudflare Tunnel |
| 4 | Chạy thử nghiệm pipeline 2D sang 3D (Đơn ảnh hoặc Đa ảnh) và kiểm tra file `.glb` |


In [ ]:
# Cell 1: Clone repo (nhánh P6-FullStack-Cloud) + deps + Depth-Anything-V2 + TripoSR
# Colab đã có torch GPU sẵn — KHÔNG cài lại (đè bản Colab -> vỡ CUDA runtime).
import os, sys, shutil

os.chdir('/content')   # kernel có thể đang đứng trong thư mục đã bị xoá -> mọi lệnh ! sau đó fail 'getcwd'

REPO = '/content/Img2d-to-3d'
BRANCH = os.environ.get('REPO_BRANCH', 'P6-FullStack-Cloud')
if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)      # dọn thư mục rỗng còn sót từ lần chạy trước
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO} || !git clone -q https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q {BRANCH} 2>/dev/null || true
    !git -C {REPO} reset --hard origin/{BRANCH}
os.chdir(REPO)
!git log --oneline -1
!ls notebook/backend/app.py

!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx "scikit-image<0.26.0" opencv-python-headless kornia transformers xatlas roma einops safetensors matplotlib tqdm fast-simplification
import trimesh
print(f'✅ trimesh {trimesh.__version__} OK')

# ── Depth-Anything-V2: Dự đoán độ sâu đa góc & đơn ảnh (NVIDIA TSDF reference) ──
try:
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
    print('✅ Depth-Anything-V2 OK -> Pipeline sẵn sàng cho cả Đơn ảnh & Đa ảnh 360°')
except Exception as e:
    print('⚠️ Transformers chưa sẵn sàng:', e)

# ── TripoSR (chỉ cần cho chế độ 1 ảnh / nhánh Quality FAIL) ──
if not os.path.isdir('/content/TripoSR/.git'):
    shutil.rmtree('/content/TripoSR', ignore_errors=True)
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
# KHÔNG chạy requirements.txt của TripoSR: ghim transformers==4.35.0 (đè bản RMBG-2.0 cần)
!pip install -q omegaconf einops imageio
!pip install -q torchmcubes 2>/dev/null || echo 'torchmcubes thiếu (Colab là Python 3.13) -> TripoSR tự CPU fallback'
os.system(f'rm -rf {REPO}/notebook/backend/tsr && cp -r /content/TripoSR/tsr {REPO}/notebook/backend/tsr')

# ── RMBG-2.0 (tách nền) — repo GATED, cần token HuggingFace ──
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN đã nạp -> P1 tách nền thật bằng RMBG-2.0')
except Exception:
    print('ℹ️ Không có HF_TOKEN -> P1 tách nền bằng rembg offline (u2net). Vẫn chạy bình thường.')

print('Deps OK | cwd =', os.getcwd())


In [ ]:
# Cell 2: Ảnh test — Chọn bộ dữ liệu để thử nghiệm:
# Tùy chọn 1: 'gso'       -> 5 ảnh chụp chiếc giày trên bàn (ảnh thật ngoài đời)
# Tùy chọn 2: 'objaverse' -> 6 ảnh 360° quả táo Objaverse-1k (có cả góc trên đỉnh & góc dưới đáy)
DATASET_CHOICE = 'objaverse'   # Đổi thành 'gso' nếu muốn thử lại chiếc giày

import os, json, glob, shutil

os.chdir('/content/Img2d-to-3d')
INPUT_DIR = 'input'
os.makedirs(INPUT_DIR, exist_ok=True)
# Dọn dẹp ảnh cũ trong input/
for f in glob.glob(f'{INPUT_DIR}/*'):
    if os.path.isfile(f) and not f.endswith('.gitkeep'):
        try: os.remove(f)
        except: pass

if DATASET_CHOICE == 'objaverse':
    OUT = 'data/objaverse_apple'
    print('🍎 Đang dùng 6 ảnh 360° toàn diện từ dxgl/objaverse-1k (có cả góc trên & góc đáy)')
else:
    OUT = 'data/gso'
    print('👟 Đang dùng 5 ảnh chụp vật thật chiếc giày từ Google Scanned Objects')

have = sorted(glob.glob(f'{OUT}/view_*.jpg') + glob.glob(f'{OUT}/view_*.png'))
assert len(have) >= 4, f'Không tìm thấy đủ ảnh trong {OUT}/'
print(f'✅ Đã nạp {len(have)} ảnh từ {OUT}/:', [os.path.basename(p) for p in have])

# Đồng bộ ảnh sang thư mục input/ để Web UI và Backend phục vụ đồng bộ
for f in have:
    shutil.copy2(f, os.path.join(INPUT_DIR, os.path.basename(f)))
if os.path.exists(f'{OUT}/cameras.json'):
    shutil.copy2(f'{OUT}/cameras.json', os.path.join(INPUT_DIR, 'cameras.json'))
    print('✅ Đã đồng bộ cameras.json chuẩn góc tọa độ từ dataset')
print('Đã đồng bộ ảnh sang thư mục input/:', sorted(os.listdir(INPUT_DIR)))
print('cwd =', os.getcwd())


In [ ]:
# Cell 3: Bật FastAPI + Cloudflare Tunnel (KHÔNG cần tài khoản Cloudflare)
import subprocess, time, os, re, urllib.request, sys

REPO    = '/content/Img2d-to-3d'
BACKEND = os.path.join(REPO, 'notebook', 'backend')
assert os.path.isdir(BACKEND), f'Không thấy {BACKEND} -> chạy lại Cell 1'
os.chdir(REPO)

# Dọn dẹp tiến trình server cũ
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

env = dict(os.environ)
env['PYTHONPATH'] = BACKEND + os.pathsep + env.get('PYTHONPATH', '')
env.setdefault('TSDF_RES', '128')

print('🚀 Đang khởi chạy máy chủ FastAPI Backend (app.py)...')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT, text=True,
)

# Chờ server sẵn sàng (Health check có log tiến độ trực quan, không bị treo âm thầm)
ready = False
for i in range(45):
    if server.poll() is not None:
        print('❌ Server thoát bất ngờ với exit code:', server.returncode)
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            print('✅ Server READY (Khởi động xong trong ' + str(i * 2) + 's):', r.read().decode())
            ready = True
            break
    except Exception:
        if i % 3 == 0 and i > 0:
            print(f'   ...đang nạp engine ({i * 2}s)...')
        time.sleep(2)

if not ready:
    print('⚠️ Server chưa phản hồi. Log máy chủ (/content/server.log):')
    if os.path.exists('/content/server.log'):
        print(open('/content/server.log', encoding='utf-8', errors='replace').read()[-3000:])

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT, text=True,
)
url = None
for _ in range(30):
    if os.path.exists('/content/tunnel.log'):
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', open('/content/tunnel.log', encoding='utf-8', errors='replace').read())
        if m:
            url = m.group(0)
            print('\n🌐 WEB UI  :', url)
            print('📘 SWAGGER :', url + '/docs')
            break
    time.sleep(1)

if url is None:
    print('Không lấy được URL tunnel — xem /content/tunnel.log')


In [ ]:
# Cell 4: Chạy thử pipeline + in TOÀN BỘ log P1→P5
# Gọi thẳng 127.0.0.1 (không qua tunnel) nên chạy bao lâu cũng được — dùng được endpoint sync.
# Đổi TEST_SINGLE_VIEW = True nếu muốn thử kịch bản 1 ảnh duy nhất.
TEST_SINGLE_VIEW = False

import glob, subprocess, json, os

os.chdir('/content/Img2d-to-3d')

imgs = sorted(glob.glob('/content/Img2d-to-3d/input/view_*.jpg') + glob.glob('/content/Img2d-to-3d/input/view_*.png'))
assert imgs, 'Không có ảnh test -> chạy lại Cell 2'

if TEST_SINGLE_VIEW:
    print('🚀 Đang chạy Kịch bản ĐƠN ẢNH (1 ảnh) qua Depth-Anything-V2:', os.path.basename(imgs[0]))
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/', '-F', f'files=@{imgs[0]}']
else:
    print(f'🚀 Bắt đầu gửi request tới API (quá trình xử lý mất ~5–10 giây, xin vui lòng đợi)...\n🚀 Đang chạy Kịch bản ĐA ẢNH 360° với {len(imgs[:8])} ảnh qua NVIDIA TSDF + Depth-Anything-V2...')
    print('Danh sách ảnh:', [os.path.basename(p) for p in imgs[:8]])
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/']
    for p in imgs[:8]:
        cmd += ['-F', f'files=@{p}']

r = subprocess.run(cmd, capture_output=True, text=True)
print('\n--- KẾT QUẢ API ---')
try:
    print(json.dumps(json.loads(r.stdout), indent=2, ensure_ascii=False))
except Exception:
    print('stdout:', r.stdout[:1000], '\nstderr:', r.stderr[-500:])

print('\n--- FILE .glb ĐÃ XUẤT ---')
all_glbs = sorted(set(
    glob.glob('/content/Img2d-to-3d/output/*.glb') +
    glob.glob('/content/Img2d-to-3d/notebook/backend/outputs/*.glb')
))
for f in all_glbs:
    print(f'{os.path.basename(f)}  {os.path.getsize(f):,} bytes')

print('\n--- LOG SERVER (P1→P5) — copy từ đây nếu cần báo lỗi ---')
print(open('/content/server.log', encoding='utf-8', errors='replace').read()[-8000:])


In [ ]:
# (BỎ COMMENT rồi chạy cell này) — xem 40 vật đầu để chọn OBJ_INDEX cho Cell 2
# from datasets import load_dataset
# ds = load_dataset("suvadityamuk/google-scanned-objects", split="train", streaming=True)
# for i, s in enumerate(ds):
#     if i >= 40: break
#     m = s["json"]
#     print(i, (m.get('category') or m.get('category_name')), '|', m.get('name'))


## Dừng server

**Runtime ▸ Restart session** (hoặc chạy `!pkill -f uvicorn`). File `.glb` nằm ở
`output/` — tải về bằng panel **Files** bên trái.

## Nguồn ảnh khác

| Nguồn | Đặc điểm |
| :--- | :--- |
| **Google Scanned Objects** (Cell 2) | 1030 vật / 17 nhóm, 5 ảnh render + `gt.glb` để so |
| OmniObject3D | 6000 vật / 190 nhóm, **100 ảnh 800×800** mỗi vật (cần đăng ký OpenDataLab) |
| CO3D (Meta) | 19k vật, ảnh chụp thật ngoài đời, có mask tách nền sẵn (bản nhỏ 8.9 GB) |
| DTU | Benchmark MVS, 49–64 góc, có ground-truth point cloud |

**Tốt nhất vẫn là ảnh tự chụp**: đặt vật lên nền trơn → đi vòng quanh, **8 tấm cách đều ~45°**,
giữ nguyên khoảng cách và độ cao máy, **không xoay vật**.
